# Assignment No. 4 – Machine Learning Model Application & Evaluation
## Student Academic Risk Prediction

**Student:** Ali Raza  
**Roll No.:** 014  
**Section:** BSCS 3A  
**Problem Type:** Binary Classification

Compare Logistic Regression, Decision Tree and Random Forest using required classification metrics. Dataset: UCI Student Performance. Target: at_risk = 1 when G3 < 10, otherwise 0. G1, G2 and G3 are excluded from features to prevent leakage.

## 1. Import libraries and load dataset

In [ ]:
import io, zipfile, urllib.request
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

url = 'https://archive.ics.uci.edu/static/public/320/student+performance.zip'
with urllib.request.urlopen(url, timeout=60) as response:
    zip_bytes = response.read()
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    math_df = pd.read_csv(z.open('student-mat.csv'), sep=';')
    portuguese_df = pd.read_csv(z.open('student-por.csv'), sep=';')
df = pd.concat([math_df, portuguese_df], ignore_index=True).drop_duplicates().reset_index(drop=True)
print('Dataset shape:', df.shape)
display(df.head())

## 2. Prepare target and split data

In [ ]:
df['at_risk'] = (df['G3'] < 10).astype(int)
print('Target distribution (0=Not at risk, 1=At risk):')
display(df['at_risk'].value_counts())
df['at_risk'].value_counts().sort_index().plot(kind='bar', title='Student Risk Class Distribution')
plt.xlabel('Class (0 = Not at risk, 1 = At risk)'); plt.ylabel('Records'); plt.tight_layout(); plt.show()

X = df.drop(columns=['G1', 'G2', 'G3', 'at_risk'])
y = df['at_risk']
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()
preprocessor = ColumnTransformer(transformers=[
 ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
 ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Training records:', len(X_train), '| Testing records:', len(X_test))

## 3. Train and evaluate three models

In [ ]:
models = {
 'Logistic Regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
 'Decision Tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
 'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=10, class_weight='balanced', random_state=42)
}
fitted_models, predictions, results = {}, {}, []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted_models[name], predictions[name] = pipe, pred
    results.append({'Model': name, 'Accuracy': accuracy_score(y_test, pred), 'Precision': precision_score(y_test, pred, zero_division=0), 'Recall': recall_score(y_test, pred, zero_division=0), 'F1-Score': f1_score(y_test, pred, zero_division=0)})
results_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False).reset_index(drop=True)
display(results_df.round(3))

## 4. Confusion matrices and classification reports

In [ ]:
for name, pred in predictions.items():
    print('\n' + '=' * 55 + '\n' + name)
    print(classification_report(y_test, pred, target_names=['Not at risk', 'At risk'], zero_division=0))
    disp = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix(y_test, pred), display_labels=['Not at risk', 'At risk'])
    disp.plot(values_format='d')
    plt.title(f'Confusion Matrix — {name}')
    plt.tight_layout()
    plt.show()

## 5. Model comparison and conclusion

In [ ]:
results_df.set_index('Model')[['Accuracy', 'Precision', 'Recall', 'F1-Score']].plot(kind='bar', ylim=(0, 1), figsize=(10, 5))
plt.title('Model Performance Comparison'); plt.ylabel('Score'); plt.xticks(rotation=15, ha='right'); plt.tight_layout(); plt.show()
best = results_df.iloc[0]
print('Best model by F1-score:', best['Model'])
print(best.to_string())

print('\nConclusion: The model with the highest measured F1-score is selected as the best overall model. F1 balances precision and recall. Review recall carefully when the goal is to avoid missing students who may need support.')

## Dataset citation and limitations
Cortez, P. & Silva, A. (2008). *Using data mining to predict secondary school student performance*. UCI Machine Learning Repository: https://archive.ics.uci.edu/dataset/320/student+performance

This dataset is an educational example and may not represent all universities. The threshold G3 < 10 is a chosen definition, not a universal standard. Any real use requires local validation, privacy safeguards, fairness checks and human review.